# Mapa de IPM - Indice de Pobreza Multidimensional (Ciudad Paraiso)

Mapa interactivo del **IPM por manzana** recortado a Ciudad Paraiso. Fuente:
`data/geojson/Mzn_ipm_variables.geojson` (15.658 manzanas de Cali; se filtran las que
estan dentro del poligono de la zona).

- **IPM** (indice 0-100): coropleta por manzana. Mayor = mayor pobreza multidimensional.
- **15 variables de privacion**: analfabetismo, bajo logro educativo, inasistencia,
  rezago, trabajo infantil, dependencia economica, empleo informal, barreras en salud,
  sin aseguramiento, hacinamiento, paredes/pisos inadecuados, sin alcantarillado, sin agua.

Estandares del proyecto: paleta Okabe-Ito, capas base Google (sin OpenStreetMap),
mascara, capas transversales ITT, panel colapsable con grafica de privaciones y boton
unico 'Exportar lamina (deck)'. Ejecutar en el entorno uv unico del proyecto.


## 0. Setup Colab (opcional)

En Google Colab esta celda instala dependencias, clona el repo y descarga los datos
publicos. En el entorno uv local no hace nada.

In [ ]:
# === SETUP COLAB (se ignora en local) ===
# En Google Colab: instala dependencias, clona el repo y descomprime el ZIP de
# datos PUBLICOS (sin PII) a data/1_data_itt/. En el entorno uv local NO hace nada.
def _en_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except Exception:
        return False
if _en_colab():
    import os, subprocess, sys, zipfile
    REPO = "https://github.com/j0rg3c45/mapa_Analitico_ciudad_Paraiso.git"
    DEST = "/content/mapa_Analitico_ciudad_Paraiso"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "geopandas", "folium", "mapclassify", "pyarrow", "openpyxl"], check=False)
    if not os.path.isdir(DEST):
        subprocess.run(["git", "clone", "--depth", "1", REPO, DEST], check=False)
    _zip = os.path.join(DEST, "data_publico", "data_itt_publico.zip")
    _data_dir = os.path.join(DEST, "data")
    if os.path.isfile(_zip):
        os.makedirs(_data_dir, exist_ok=True)
        with zipfile.ZipFile(_zip) as z:
            z.extractall(_data_dir)
        print("Datos publicos descomprimidos en:", os.path.join(_data_dir, "1_data_itt"))
    else:
        print("Aviso: no se encontro", _zip, "- revisa que el repo lo incluya.")
    _nb_dir = os.path.join(DEST, "notebooks")
    if os.path.isdir(_nb_dir):
        os.chdir(_nb_dir)
    print("Entorno Colab listo. Directorio de trabajo:", os.getcwd())
else:
    print("Entorno local (uv). Setup Colab omitido.")

## 1. Entorno y paleta Okabe-Ito

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import geopandas as gpd
OKI_AZUL="#0072B2"; OKI_NARANJA="#E69F00"; OKI_VERDE="#009E73"
OKI_AMARILLO="#F0E442"; OKI_AZUL_C="#56B4E9"; OKI_BERMELL="#D55E00"
OKI_MORADO="#CC79A7"; OKI_GRIS="#333333"
BG="#F4F6F9"
CRS_WEB=4326; CRS_METRIC=9377
print("Entorno listo. Paleta Okabe-Ito cargada.")

## 2. Rutas y poligono de la zona

In [ ]:
def _resolver_poligono():
    env=os.environ.get("POLIGONO_ZONA"); cand=[env] if env else []
    cand+=[os.path.join("..","data","geojson","poligono_ciudad_Paraiso.geojson"),
           os.path.join("data","geojson","poligono_ciudad_Paraiso.geojson"),
           "/content/data/geojson/poligono_ciudad_Paraiso.geojson",
           "/content/drive/MyDrive/mapa_Analitico_ciudad_Paraiso/data/geojson/poligono_ciudad_Paraiso.geojson"]
    for p in cand:
        if p and os.path.isfile(p): return os.path.abspath(p)
    return None
_poly_path=_resolver_poligono()
POLIGONO_ZONA=None
if _poly_path:
    POLIGONO_ZONA=gpd.read_file(_poly_path).to_crs(CRS_WEB)
    print("Poligono de zona:",_poly_path,"| features:",len(POLIGONO_ZONA))
else:
    print("Aviso: sin poligono de zona.")
OUT_DIR=None
for _base in [os.path.join("..","data_processed_geo"),"data_processed_geo","/content/data_processed_geo"]:
    try:
        os.makedirs(_base,exist_ok=True); OUT_DIR=os.path.abspath(_base); break
    except Exception: continue
print("Salida procesada:", OUT_DIR)
# Carpeta de salida de MAPAS HTML: mapa_html/ (junto a data_processed_geo/).
MAPAS_DIR = os.path.abspath(os.path.join(os.path.dirname(OUT_DIR.rstrip(os.sep)), "mapa_html"))
os.makedirs(MAPAS_DIR, exist_ok=True)
print("Salida de mapas HTML:", MAPAS_DIR)
def _ruta_itt(nombre):
    cand=[os.path.join("..","data","1_data_itt",nombre),
          os.path.join("data","1_data_itt",nombre),
          os.path.join("/content","data","1_data_itt",nombre),
          os.path.join("/content","drive","MyDrive","mapa_Analitico_ciudad_Paraiso","data","1_data_itt",nombre)]
    for p in cand:
        if os.path.isfile(p): return os.path.abspath(p)
    return None
def _ruta_geojson(nombre):
    cand=[os.path.join("..","data","geojson",nombre),
          os.path.join("data","geojson",nombre),
          os.path.join("/content","data","geojson",nombre),
          os.path.join("/content","drive","MyDrive","mapa_Analitico_ciudad_Paraiso","data","geojson",nombre)]
    for p in cand:
        if os.path.isfile(p): return os.path.abspath(p)
    return None

## 3. Carga del IPM y recorte a la zona

In [ ]:
# 3a. IPM por manzana (Cali) recortado a los poligonos dentro de Ciudad Paraiso.
_rp_ipm=_ruta_geojson("IPM_ciudad_Paraiso.geojson")
if _rp_ipm is None:
    raise FileNotFoundError("No se encontro IPM_ciudad_Paraiso.geojson en data/geojson.")
_raw=gpd.read_file(_rp_ipm)
_raw=_raw.set_crs(epsg=CRS_WEB) if _raw.crs is None else _raw.to_crs(CRS_WEB)
_raw=_raw[_raw.geometry.notna()].copy()
_raw["ipm"]=pd.to_numeric(_raw["ipm"], errors="coerce")
# Filtro espacial: manzanas DENTRO del poligono de la zona (point-in-polygon real).
if POLIGONO_ZONA is not None:
    _zona=POLIGONO_ZONA.geometry.union_all()
    ipm_z=_raw[_raw.geometry.within(_zona) | _raw.geometry.intersects(_zona)].copy()
else:
    ipm_z=_raw.copy()
ipm_z=ipm_z.reset_index(drop=True)
print("Manzanas IPM (total Cali):",len(_raw),"| en la zona:",len(ipm_z))
print("IPM en zona -> min: %.1f  max: %.1f  promedio: %.1f" % (
      ipm_z["ipm"].min(), ipm_z["ipm"].max(), ipm_z["ipm"].mean()))

## 4. Exportacion de lamina (deck) - helper reutilizable

In [ ]:
# Helper reutilizable para inyectar en un mapa Folium UN boton de exportacion
# de lamina con estructura de deck (PMU). Captura el contenedor del mapa como
# imagen y la compone dentro de una lamina HTML (encabezado + 2 columnas +
# pie) que se exporta a PDF (una pagina). Este archivo es solo una plantilla
# de trabajo; su contenido se inserta como celda en los notebooks.
#
# Decision CORS/tiles: html2canvas no puede rasterizar los tiles base de
# Google/Esri por restricciones CORS (saldrian en blanco o rotos). Por eso,
# antes de capturar el mapa se oculta el panel de tiles base (.leaflet-tile-pane)
# y se pinta un fondo claro (#F4F6F9) en el contenedor; se capturan solo los
# overlays vectoriales (contorno, manzanas, puntos, calor). La imagen del mapa
# va en la columna derecha de la lamina.
def agregar_export(mapa, titulo, slug):
    """Inyecta en 'mapa' (folium.Map) un control flotante con UN boton:
    'Exportar lamina (deck)'. Al pulsar, captura el contenedor del mapa a una
    imagen y construye una lamina HTML con estructura de deck del PMU
    (encabezado dorado, 2 columnas: resumen/cifras clonadas del panel +
    mapa, pie con corte y fuente); captura la lamina y la exporta a PDF de
    una pagina (landscape) con dialogo "Guardar como" (showSaveFilePicker) y
    fallback a descarga. Nombre: deck_<slug>_<AAAA-MM-DD>.pdf. Debe llamarse
    ANTES de mapa.save(...).
    - titulo: titulo visible en el encabezado de la lamina.
    - slug: nombre corto para el archivo.
    """
    map_id = mapa.get_name()  # id del div .folium-map de ESTE mapa
    # Librerias por CDN (sin build).
    cdn = (
        '<script src="https://cdn.jsdelivr.net/npm/html2canvas@1.4.1/dist/html2canvas.min.js"></script>'
        '<script src="https://cdn.jsdelivr.net/npm/jspdf@2.5.1/dist/jspdf.umd.min.js"></script>'
    )
    mapa.get_root().header.add_child(folium.Element(cdn))
    # Control + logica. Se escapa como string de Python; en JS se usa map_id,
    # titulo y slug concretos de ESTE mapa (asi funciona con varios mapas).
    titulo_js = titulo.replace("\\", "\\\\").replace("'", "\\'")
    slug_js = slug.replace("\\", "\\\\").replace("'", "\\'")
    bloque = """
<div id="exportctl___MID__" style="position:fixed;top:12px;right:12px;z-index:10000;
     display:flex;flex-direction:column;gap:6px;font-family:system-ui,Segoe UI,sans-serif">
  <button id="btndeck___MID__" style="cursor:pointer;border:1px solid #333;background:#fff;
     color:#222;padding:6px 10px;border-radius:6px;box-shadow:0 2px 6px rgba(0,0,0,.25);
     font-size:12px;font-weight:600">Exportar lamina (deck)</button>
</div>
<script>
(function(){
  var MAP_ID = '__MID__';
  var TITULO = '__TITULO__';
  var SLUG = '__SLUG__';
  var FUENTE = 'ITT Ciudad Paraiso - CDO Alcaldia de Santiago de Cali';
  var DORADO = '#8A6D1A';
  function fechaISO(){
    var d = new Date();
    var mm = ('0'+(d.getMonth()+1)).slice(-2);
    var dd = ('0'+d.getDate()).slice(-2);
    return d.getFullYear()+'-'+mm+'-'+dd;
  }
  function getContainer(){
    return document.getElementById(MAP_ID);
  }
  // Paso (a): captura SOLO el contenedor del mapa a un canvas, ocultando antes
  // los botones de export, el selector de capas, los tiles base (CORS), el
  // panel flotante de graficas y el slider de intensidad del calor; restaura
  // todo despues. Devuelve el canvas via callback.
  function capturarMapa(cont, cb){
    var ctl = document.getElementById('exportctl_'+MAP_ID);
    if(ctl) ctl.style.display = 'none';
    var layerCtls = cont.querySelectorAll('.leaflet-control-layers');
    layerCtls.forEach(function(lc){ lc.dataset.prevDisplay = lc.style.display; lc.style.display = 'none'; });
    // Panel de graficas flotante (#seg-panel o #eu-panel) y slider del calor.
    var panel = cont.querySelector('#seg-panel, #eu-panel, #ipm-panel');
    var prevPanelDisplay = null;
    if(panel){ prevPanelDisplay = panel.style.display; panel.style.display = 'none'; }
    var slider = cont.querySelector('#heat-intensity-ctrl');
    var prevSliderDisplay = null;
    if(slider){ prevSliderDisplay = slider.style.display; slider.style.display = 'none'; }
    // CORS: ocultar tiles base y aplicar fondo claro.
    var tilePanes = cont.querySelectorAll('.leaflet-tile-pane');
    var fondoPrev = cont.style.background;
    cont.style.background = '#F4F6F9';
    tilePanes.forEach(function(tp){ tp.dataset.prevOpacity = tp.style.opacity; tp.style.opacity = '0'; });
    function restaurar(){
      if(ctl) ctl.style.display = '';
      layerCtls.forEach(function(lc){ lc.style.display = ''; });
      if(panel) panel.style.display = (prevPanelDisplay || '');
      if(slider) slider.style.display = (prevSliderDisplay || '');
      tilePanes.forEach(function(tp){ tp.style.opacity = tp.dataset.prevOpacity || ''; });
      cont.style.background = fondoPrev;
    }
    html2canvas(cont, {useCORS:true, backgroundColor:'#F4F6F9', logging:false, scale:2})
      .then(function(canvas){ restaurar(); cb(null, canvas); })
      .catch(function(err){ restaurar(); cb(err, null); });
  }
  // Paso previo: asegura el panel #seg-panel/#eu-panel expandido (si usa
  // <details>) para que Chart.js tenga el canvas visible y pintado, luego
  // captura el canvas real del grafico a un dataURL PNG. Devuelve null si
  // no hay canvas o esta vacio. cb(dataUrl|null).
  function capturarGrafico(cb){
    var panel = document.querySelector('#seg-panel, #eu-panel, #ipm-panel');
    if(!panel){ cb(null); return; }
    var dets = panel.querySelectorAll('details');
    dets.forEach(function(d){ d.setAttribute('open', 'open'); });
    // Pequeno tiempo para que el canvas quede visible y Chart.js pinte.
    setTimeout(function(){
      var cv = panel.querySelector('canvas[id*="chart"]') || panel.querySelector('canvas');
      if(!cv || !cv.width || !cv.height){ cb(null); return; }
      try { cb(cv.toDataURL('image/png')); }
      catch(e){ cb(null); }
    }, 250);
  }
  // Paso (b): construye el div-lamina temporal con estructura de deck.
  // La imagen del mapa (canvas de (a)) va en la columna derecha; el panel
  // existente (si lo hay) se clona en la columna izquierda.
  function construirLamina(mapaDataUrl, graficoDataUrl){
    var lam = document.createElement('div');
    lam.id = 'deck-lamina';
    lam.style.cssText = 'position:fixed;top:0;left:0;z-index:2147483647;'+
      'width:1200px;background:#FFFFFF;color:#333;'+
      'font-family:system-ui,Segoe UI,sans-serif;'+
      'box-sizing:border-box;overflow:hidden;';
    // ENCABEZADO (barra con titulo + linea de seccion gris + borde dorado).
    var head = document.createElement('div');
    head.style.cssText = 'padding:16px 22px 12px 22px;border-bottom:3px solid '+DORADO+';';
    head.innerHTML =
      '<div style="font-size:20px;font-weight:700;color:#222;line-height:1.2">'+TITULO+'</div>'+
      '<div style="font-size:12px;color:#888;margin-top:4px">ITT Ciudad Paraiso - Lamina de deck (PMU)</div>';
    lam.appendChild(head);
    // CUERPO en 2 columnas (grid).
    var body = document.createElement('div');
    body.style.cssText = 'display:grid;grid-template-columns:35% 65%;gap:16px;'+
      'padding:16px 22px;align-items:start;';
    // Columna izquierda: resumen / cifras (clon del panel si existe).
    var colIzq = document.createElement('div');
    colIzq.style.cssText = 'min-width:0;';
    var panelOrig = document.querySelector('#seg-panel, #eu-panel, #ipm-panel');
    if(panelOrig){
      var clon = panelOrig.cloneNode(true);
      clon.removeAttribute('id');
      // Sacar el clon del flujo fijo para que fluya dentro de la columna.
      clon.style.position = 'static';
      clon.style.top = 'auto'; clon.style.left = 'auto';
      clon.style.right = 'auto'; clon.style.bottom = 'auto';
      clon.style.zIndex = 'auto';
      clon.style.maxWidth = 'none';
      clon.style.width = '100%';
      clon.style.boxShadow = 'none';
      // Expandir cualquier <details> para que la serie se vea.
      var dets = clon.querySelectorAll('details');
      dets.forEach(function(d){ d.setAttribute('open', 'open'); });
      // El canvas de Chart.js se clona VACIO (los canvas no conservan su
      // contenido al clonar). Lo sustituimos por un <img> con la imagen
      // capturada del canvas real para que el grafico SE VEA en la lamina.
      if(graficoDataUrl){
        var cvClon = clon.querySelector('canvas');
        if(cvClon){
          var imgChart = document.createElement('img');
          imgChart.src = graficoDataUrl;
          imgChart.style.cssText = 'display:block;width:100%;height:auto;';
          if(cvClon.parentNode){ cvClon.parentNode.replaceChild(imgChart, cvClon); }
        }
      }
      colIzq.appendChild(clon);
    } else {
      colIzq.innerHTML =
        '<div style="font-size:15px;font-weight:700;color:#222;margin-bottom:6px">'+TITULO+'</div>'+
        '<div style="font-size:13px;color:#666">Resumen no disponible</div>';
    }
    // Columna derecha: imagen del mapa escalada al ancho de la columna.
    var colDer = document.createElement('div');
    colDer.style.cssText = 'min-width:0;';
    var img = document.createElement('img');
    img.src = mapaDataUrl;
    img.style.cssText = 'display:block;width:100%;height:auto;'+
      'border:1px solid #d9dde3;border-radius:6px;';
    colDer.appendChild(img);
    body.appendChild(colIzq);
    body.appendChild(colDer);
    lam.appendChild(body);
    // PIE (corte + fuente + nota).
    var pie = document.createElement('div');
    pie.style.cssText = 'padding:10px 22px 14px 22px;border-top:1px solid #e2e5ea;'+
      'font-size:11px;color:#777;';
    pie.innerHTML = 'Corte: '+new Date().toLocaleString()+
      '  |  Fuente: '+FUENTE+'  |  Mapa ilustrativo';
    lam.appendChild(pie);
    document.body.appendChild(lam);
    return lam;
  }
  // Dialogo "Guardar como" (Chrome/Edge) con fallback a descarga clasica.
  function guardarPDF(blob, nombre){
    function descargaFallback(){
      var url = URL.createObjectURL(blob);
      var a = document.createElement('a');
      a.href = url; a.download = nombre;
      document.body.appendChild(a); a.click(); document.body.removeChild(a);
      setTimeout(function(){ URL.revokeObjectURL(url); }, 1500);
    }
    if(window.showSaveFilePicker){
      var tipos = [{description:'Documento PDF', accept:{'application/pdf':['.pdf']}}];
      window.showSaveFilePicker({suggestedName:nombre, types:tipos})
        .then(function(handle){ return handle.createWritable(); })
        .then(function(writable){ return writable.write(blob).then(function(){ return writable.close(); }); })
        .catch(function(e){
          // Si el usuario cancela (AbortError) no se descarga nada; otros -> fallback.
          if(e && e.name === 'AbortError'){ return; }
          descargaFallback();
        });
    } else {
      descargaFallback();
    }
  }
  // Bounds del poligono de la zona de estudio (Ciudad Paraiso) para
  // reencuadrar el mapa SOLO en la generacion de la lamina (no afecta la
  // vista interactiva previa del usuario; tras exportar queda en este
  // encuadre pero la interactividad se mantiene).
  var ZONA_BOUNDS = L.latLngBounds([[3.442445,-76.53259],[3.44791,-76.526004]]);
  function exportar(){
    var cont = getContainer();
    if(!cont){ alert('No se encontro el contenedor del mapa.'); return; }
    // 1) Forzar el panel a 'open' y capturar el grafico real a dataURL.
    capturarGrafico(function(graficoDataUrl){
      var mp = window[MAP_ID];
      // 2) fitBounds al poligono. maxZoom ENTERO evita el zoom fraccional que
      //    descoloca las capas en html2canvas (desfase al imprimir).
      var capturado = false;
      function trasEncuadre(){
        if(capturado){ return; }
        capturado = true;
        try { if(mp){ mp.invalidateSize(false); } } catch(e){}
        // 3) Margen corto tras asentar panes/tiles, luego capturar.
        setTimeout(function(){
        // 4) Capturar el mapa (ocultando botones/tiles/panel/slider).
        capturarMapa(cont, function(err, canvasMapa){
          if(err){ alert('Error al capturar el mapa: '+err); return; }
          var mapaDataUrl = canvasMapa.toDataURL('image/jpeg', 0.95);
          // 5) Construir lamina (cifras + grafico como imagen + mapa).
          var lam = construirLamina(mapaDataUrl, graficoDataUrl);
          function limpiar(){
            if(lam && lam.parentNode) lam.parentNode.removeChild(lam);
          }
          // 6) Capturar la lamina y generar el PDF.
          html2canvas(lam, {backgroundColor:'#FFFFFF', logging:false, scale:2, useCORS:true})
            .then(function(canvas){
              limpiar();
              var imgData = canvas.toDataURL('image/jpeg', 0.95);
              var jsPDFns = (window.jspdf && window.jspdf.jsPDF) ? window.jspdf.jsPDF : window.jsPDF;
              var w = canvas.width, hh = canvas.height;
              var orient = (w >= hh) ? 'landscape' : 'portrait';
              var doc = new jsPDFns({orientation:orient, unit:'pt', format:[w, hh]});
              doc.addImage(imgData, 'JPEG', 0, 0, w, hh);
              var blob = doc.output('blob');
              var nombre = 'deck_'+SLUG+'_'+fechaISO()+'.pdf';
              guardarPDF(blob, nombre);
            })
            .catch(function(e){
              limpiar();
              alert('Error al exportar la lamina: '+e);
            });
        });
        }, 350);
      }
      // Disparar fitBounds y esperar 'moveend'; fallback por timeout si no llega.
      try {
        if(mp){
          mp.once('moveend', trasEncuadre);
          mp.fitBounds(ZONA_BOUNDS, {padding:[10,10], maxZoom: 17, animate: false});
          setTimeout(trasEncuadre, 1200);
        } else {
          trasEncuadre();
        }
      } catch(e){ trasEncuadre(); }
    });
  }
  function init(){
    var b = document.getElementById('btndeck_'+MAP_ID);
    if(b) b.addEventListener('click', function(){ exportar(); });
  }
  if(document.readyState === 'loading'){
    document.addEventListener('DOMContentLoaded', init);
  } else { init(); }
})();
</script>
"""
    bloque = (bloque.replace("__MID__", map_id)
                    .replace("__TITULO__", titulo_js)
                    .replace("__SLUG__", slug_js))
    mapa.get_root().html.add_child(folium.Element(bloque))
    return mapa

## 5. Construccion del mapa interactivo (Folium)

In [ ]:
# 5a. Mapa Folium del IPM: coropleta del indice de pobreza multidimensional por manzana.
#      Agregado por manzana (dato censal), rampa por valor de IPM.
import folium
# Escala IPM de 4 niveles (referencia Colombia urbano) como mapa tipo calor.
# Rango 0-100 (el IPM es un porcentaje). Rampa verde->amarillo->naranja->rojo.
IPM_NIVELES = [
    (0.0, 8.0,   "Muy bajo / Bajo",       "#1a9850"),
    (8.1, 15.0,  "Medio / Tipico urbano", "#a6d96a"),
    (15.1, 25.0, "Alto",                  "#fdae61"),
    (25.1, 100.0,"Critico / Severo",      "#d73027"),
]
def _niv_idx(v):
    v = float(v)
    for i, (lo, hi, _n, _c) in enumerate(IPM_NIVELES):
        if v <= hi:
            return i
    return len(IPM_NIVELES) - 1
def _cat_nombre(v):
    return IPM_NIVELES[_niv_idx(v)][2]
def _color_ipm(v):
    return IPM_NIVELES[_niv_idx(v)][3]
# Centro del mapa.
if POLIGONO_ZONA is not None:
    _c = POLIGONO_ZONA.geometry.union_all().centroid
    centro = [_c.y, _c.x]
else:
    centro = [ipm_z.geometry.union_all().centroid.y, ipm_z.geometry.union_all().centroid.x]
m_ipm = folium.Map(location=centro, zoom_start=16, tiles=None, control_scale=True,
                   zoom_snap=0.25, zoom_delta=0.1, max_zoom=22)
m_ipm.get_root().width = "100%"
m_ipm.get_root().height = "100%"
_css_full = ("<style>html,body{height:100%;margin:0;padding:0}"
             ".folium-map{position:absolute;top:0;bottom:0;left:0;right:0;"
             "width:100%;height:100%}</style>")
m_ipm.get_root().header.add_child(folium.Element(_css_full))
# Capas base conmutables (Google Streets activa). Sin OpenStreetMap.
folium.TileLayer(tiles="https://mt1.google.com/vt/lyrs=m&x={x}&y={y}&z={z}",
    attr="Google", name="Google Streets", show=True, max_zoom=22, max_native_zoom=20).add_to(m_ipm)
folium.TileLayer(tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
    attr="Esri", name="Esri Satelite", show=False, max_zoom=22, max_native_zoom=20).add_to(m_ipm)
folium.TileLayer(tiles="https://mt1.google.com/vt/lyrs=s&x={x}&y={y}&z={z}",
    attr="Google", name="Google Satelite", show=False, max_zoom=22, max_native_zoom=20).add_to(m_ipm)
# Mascara oscura fuera de la zona (foco).
if POLIGONO_ZONA is not None:
    _zona_geom = POLIGONO_ZONA.geometry.union_all()
    _mundo = [[-180.0, -85.0], [180.0, -85.0], [180.0, 85.0], [-180.0, 85.0], [-180.0, -85.0]]
    _agujeros = []
    _geoms = list(_zona_geom.geoms) if _zona_geom.geom_type == "MultiPolygon" else [_zona_geom]
    for _g in _geoms:
        _agujeros.append([[x, y] for x, y in _g.exterior.coords])
    _mascara = {"type": "Feature", "properties": {},
                "geometry": {"type": "Polygon", "coordinates": [_mundo] + _agujeros}}
    folium.GeoJson(_mascara, name="Mascara (foco en la zona)",
        style_function=lambda f: {"fillColor": "#000000", "color": "#000000",
                                  "weight": 0, "fillOpacity": 0.25},
        control=False).add_to(m_ipm)
# Limite oficial de la zona (borde dorado punteado). Activo.
_fg_limite = None
_rp_lim = _ruta_itt("00_limite_ciudad_paraiso.geojson")
if _rp_lim is not None:
    _lim = gpd.read_file(_rp_lim).to_crs(CRS_WEB)
    _fg_limite = folium.FeatureGroup(name="Limite Ciudad Paraiso", show=True)
    folium.GeoJson(_lim.to_json(),
        style_function=lambda f: {"color": "#8A6D1A", "weight": 5, "dashArray": "10 6", "fill": False}
    ).add_to(_fg_limite)
# --- Capas del dataset ITT consolidado (data/1_data_itt), conmutables y apagadas ---
def _ruta_itt(nombre):
    cands = [
        __import__("os").path.join("..", "data", "1_data_itt", nombre),
        __import__("os").path.join("data", "1_data_itt", nombre),
        __import__("os").path.join("/content", "data", "1_data_itt", nombre),
        __import__("os").path.join("/content", "drive", "MyDrive",
                   "mapa_Analitico_ciudad_Paraiso", "data", "1_data_itt", nombre),
    ]
    for _p in cands:
        if __import__("os").path.isfile(_p):
            return __import__("os").path.abspath(_p)
    return None
def _add_geojson_itt(nombre, capa_nombre, style, tooltip_field=None, popup_fn=None, to_points=False, show=False):
    _rp = _ruta_itt(nombre)
    if _rp is None:
        print("Aviso: no se encontro", nombre, "- capa omitida")
        return None
    _g = gpd.read_file(_rp).to_crs(CRS_WEB)
    # Convertir columnas de fecha/timestamp a texto (folium no serializa Timestamp).
    for _col in _g.columns:
        if _col == "geometry":
            continue
        if str(_g[_col].dtype).startswith("datetime"):
            _g[_col] = _g[_col].astype(str)
    _fg = folium.FeatureGroup(name=capa_nombre, show=show)
    if to_points:
        for _, _r in _g.iterrows():
            if _r.geometry is None:
                continue
            _pt = _r.geometry
            folium.CircleMarker(
                location=[float(_pt.y), float(_pt.x)], radius=4,
                color="#333333", weight=1, fill=True, fill_color="#D55E00", fill_opacity=0.75,
                popup=(folium.Popup(popup_fn(_r), max_width=240) if popup_fn else None),
            ).add_to(_fg)
    else:
        folium.GeoJson(
            _g.to_json(),
            style_function=(lambda f, s=style: s),
            tooltip=(folium.GeoJsonTooltip(fields=[tooltip_field]) if tooltip_field and tooltip_field in _g.columns else None),
        ).add_to(_fg)
    print("Capa ITT agregada:", capa_nombre, "(", len(_g), "features )")
    return _fg
def _popup_hom300(r):
    f = r.get("fecha")
    try:
        f = f.strftime("%Y-%m-%d")
    except Exception:
        f = str(f)
    d = r.get("dist_a_estacion_m")
    arma = str(r.get("tipo_arma") or "s/d")
    movil = str(r.get("movil") or "s/d")
    return ('<div style="font-family:system-ui,Segoe UI,sans-serif;font-size:12px;'
            'background:%s;padding:6px 8px;border-radius:6px;min-width:170px">'
            '<div style="font-weight:700;color:#D55E00">Homicidio (&lt;=300 m estacion)</div>'
            '<div>Fecha: %s</div><div>Distancia: %s m</div>'
            '<div>Arma: %s</div><div>Movil: %s</div></div>'
            % (BG, f, (int(d) if d is not None else "s/d"), arma, movil))
# 01 manzanas con ITT consolidado: tooltip/popup con 4 atributos clave y area formateada.
def _add_manzanas_itt():
    _rp = _ruta_itt("01_manzanas_itt_consolidado.geojson")
    if _rp is None:
        print("Aviso: no se encontro manzanas consolidado - capa omitida")
        return None
    _g = gpd.read_file(_rp).to_crs(CRS_WEB)
    # Area metrica real en m2 (CRS metrico EPSG:9377). Si existe shape_area se ignora
    # para homogeneidad; se recalcula desde la geometria.
    _g["area_m2"] = _g.to_crs(CRS_METRIC).geometry.area
    # Formato legible: entero con separador de miles en punto (ej. 14.520 m2).
    _g["area_m2_fmt"] = _g["area_m2"].apply(lambda x: ("{:,}".format(int(round(x))).replace(",", ".")) + " m2")
    # Campos estandarizados.
    _g["id_manzana"] = _g["idmanzana"].astype(str)
    _g["barrio_f"] = _g["barrio"].astype(str)
    _g["manzana_f"] = _g["manzana"].astype(str)
    _fg = folium.FeatureGroup(name="ITT: manzanas (consolidado)", show=False)
    _tt = folium.GeoJsonTooltip(
        fields=["id_manzana", "barrio_f", "manzana_f", "area_m2_fmt"],
        aliases=["ID Manzana:", "Barrio:", "Manzana:", "Area:"],
        localize=True, sticky=True,
        style=("background-color:%s;color:%s;font-family:system-ui,Segoe UI,sans-serif;"
               "font-size:12px;padding:6px 8px;border-radius:6px;border:1px solid #ccc;" % (BG, OKI_GRIS)),
    )
    def _popup_mz(idm, bar, man, af):
        return ('<div style="font-family:system-ui,Segoe UI,sans-serif;font-size:12px;'
                'background:%s;padding:6px 8px;border-radius:6px;min-width:170px">'
                '<div style="font-weight:700;color:%s;margin-bottom:3px">Manzana %s</div>'
                '<table style="font-size:11px;border-collapse:collapse">'
                '<tr><td style="color:#666;padding:1px 6px">ID Manzana</td><td style="padding:1px 6px"><b>%s</b></td></tr>'
                '<tr><td style="color:#666;padding:1px 6px">Barrio</td><td style="padding:1px 6px">%s</td></tr>'
                '<tr><td style="color:#666;padding:1px 6px">Manzana</td><td style="padding:1px 6px">%s</td></tr>'
                '<tr><td style="color:#666;padding:1px 6px">Area</td><td style="padding:1px 6px"><b>%s</b></td></tr>'
                '</table></div>' % (BG, OKI_GRIS, man, idm, bar, man, af))
    folium.GeoJson(
        _g.to_json(),
        style_function=lambda f: {"fill": False, "color": "#333333",
                                  "weight": 1.0},
        tooltip=_tt,
    ).add_to(_fg)
    # Popups por manzana (tabla limpia).
    for _, _r in _g.iterrows():
        folium.GeoJson(
            _r["geometry"].__geo_interface__,
            style_function=lambda f: {"fillColor": "#56B4E9", "color": "#333333",
                                      "weight": 0.0, "fillOpacity": 0.0},
            popup=folium.Popup(_popup_mz(_r["id_manzana"], _r["barrio_f"],
                                         _r["manzana_f"], _r["area_m2_fmt"]), max_width=240),
        ).add_to(_fg)
    print("Capa ITT agregada: manzanas (consolidado) (", len(_g), "manzanas )")
    return _fg
# Demoliciones Cristo Rey: clasificar por estado de demolicion (columna 'demolicion').
DEMOL_COLOR = {
    "Demolido": "#D55E00",                       # bermellon: ya demolido
    "Programado para demolicion": "#E69F00",     # naranja: programado
    "En proceso (hacia demolicion)": "#F0E442",  # amarillo: en proceso
    "Sin demolicion programada": "#009E73",      # verde: conservado / sin intervencion
}
def _color_demol(v):
    return DEMOL_COLOR.get(str(v).strip(), "#999999")
def _popup_demol(r):
    est = str(r.get("demolicion") or "s/d")
    est2 = str(r.get("estado") or "")
    barrio = str(r.get("barrio") or "")
    npn = str(r.get("npn") or r.get("predio_catastral") or "")
    fdem = r.get("fecha_demolicion")
    try:
        fdem = fdem.strftime("%Y-%m-%d")
    except Exception:
        fdem = "" if fdem is None else str(fdem)
    filas = '<div style="font-weight:700;color:%s">Predio - %s</div>' % (_color_demol(est), est)
    if est2 and est2 != est:
        filas += '<div>Estado catastral: %s</div>' % est2
    if barrio:
        filas += '<div>Barrio: %s</div>' % barrio
    if npn:
        filas += '<div>Predio: %s</div>' % npn
    if fdem:
        filas += '<div>Fecha demolicion: %s</div>' % fdem
    return ('<div style="font-family:system-ui,Segoe UI,sans-serif;font-size:12px;'
            'background:%s;padding:6px 8px;border-radius:6px;min-width:180px">%s</div>'
            % (BG, filas))
def _add_demoliciones():
    _rp = _ruta_itt("10_demoliciones_cristo_rey_predios.geojson")
    if _rp is None:
        print("Aviso: no se encontro demoliciones - capa omitida")
        return None
    _g = gpd.read_file(_rp).to_crs(CRS_WEB)
    for _c in _g.columns:
        if _c != "geometry" and str(_g[_c].dtype).startswith("datetime"):
            _g[_c] = _g[_c].astype(str)
    _fg = folium.FeatureGroup(name="ITT: demoliciones Cristo Rey", show=False)
    for _, _r in _g.iterrows():
        _col = _color_demol(_r.get("demolicion"))
        folium.GeoJson(
            _r["geometry"].__geo_interface__,
            style_function=(lambda f, c=_col: {"fillColor": c, "color": "#333333",
                                               "weight": 0.8, "fillOpacity": 0.65}),
            tooltip="Demolicion: %s" % str(_r.get("demolicion") or "s/d"),
            popup=folium.Popup(_popup_demol(_r), max_width=260),
        ).add_to(_fg)
    print("Capa ITT agregada: demoliciones Cristo Rey (", len(_g), "predios )")
    return _fg
# Construccion de capas transversales (apagadas; el limite ya se agrego arriba).
_fg_manzanas = _add_manzanas_itt()
_fg_estacion = _add_geojson_itt("11_estacion_fray_damian_manzana.geojson", "ITT: estacion Fray Damian",
                                {"color": "#0072B2", "weight": 2.75, "fill": False},
                                tooltip_field="nombre")
_fg_demol = _add_demoliciones()
# Coropleta de IPM por manzana. ACTIVA al inicio.
# Etiquetas legibles de las variables de privacion para el popup.
VAR_LABEL = {
    "ANALF_": "Analfabetismo", "BAJO_": "Bajo logro educativo",
    "INFANCIA_": "Rezago infancia", "INASIS_": "Inasistencia escolar",
    "REZAGO_": "Rezago escolar", "TRAB_INFAN": "Trabajo infantil",
    "DEPEN_": "Dependencia economica", "INFOR_": "Empleo informal",
    "SALUD_": "Barreras salud", "ASEGU_": "Sin aseguramiento",
    "HACI_": "Hacinamiento", "PARED_": "Paredes inadecuadas",
    "EXCRE_": "Sin alcantarillado", "PISOS_": "Pisos inadecuados", "AGUA_": "Sin agua",
}
VARS_PRIV = list(VAR_LABEL.keys())
def _popup_ipm(r):
    # Top 3 privaciones de la manzana.
    priv = sorted([(VAR_LABEL[v], float(r.get(v) or 0.0)) for v in VARS_PRIV],
                  key=lambda x: x[1], reverse=True)[:3]
    top = "".join('<div style="font-size:11px">- %s: %.1f%%</div>' % (n, val) for n, val in priv if val > 0)
    return ('<div style="font-family:system-ui,Segoe UI,sans-serif;font-size:12px;'
            'background:%s;padding:6px 8px;border-radius:6px;min-width:200px">'
            '<div style="font-weight:700;color:#7a0177">IPM manzana</div>'
            '<div>Indice IPM: <b>%.1f</b></div>'
            '<div style="margin-top:3px;color:#666">Principales privaciones:</div>%s'
            '<div style="font-size:10px;color:#888;margin-top:3px">Fuente: IPM por manzana (DANE/censal).</div></div>'
            % (BG, float(r.get("ipm") or 0.0), top if top else '<div style="font-size:11px;color:#999">sin privaciones registradas</div>'))
_fg_ipm = folium.FeatureGroup(name="IPM: indice por manzana (" + str(len(ipm_z)) + ")", show=True)
for _, r in ipm_z.iterrows():
    _col = _color_ipm(r["ipm"])
    folium.GeoJson(r["geometry"].__geo_interface__,
        style_function=(lambda f, c=_col: {"fillColor": c, "color": "#ffffff",
                                            "weight": 0.6, "fillOpacity": 0.82}),
        tooltip="IPM: %.1f (%s)" % (float(r.get("ipm") or 0.0), _cat_nombre(r.get("ipm") or 0.0)),
        popup=folium.Popup(_popup_ipm(r), max_width=260)).add_to(_fg_ipm)
# --- Orden de overlays en el control de capas ---
_orden = [
    _fg_limite,
    _fg_manzanas,
    _fg_estacion,
    _fg_ipm,
    _fg_demol,
]
for _capa in _orden:
    if _capa is not None:
        _capa.add_to(m_ipm)
if POLIGONO_ZONA is not None:
    m_ipm.fit_bounds([[3.442445, -76.53259], [3.44791, -76.526004]], padding=(30, 30))
# Leyenda de paleta IPM (4 niveles, mapa tipo calor).
_items_ley = ""
for (lo, hi, nom, col) in IPM_NIVELES:
    _rango = ("> %s" % ("%.0f" % (lo-0.1)).replace('.', ',')) if lo > 25 else "%s - %s" % (("%.1f" % lo).replace('.', ','), ("%.1f" % hi).replace('.', ','))
    _items_ley += ('<div style="display:flex;align-items:center;gap:6px;margin:1px 0">'
                   '<i style="width:16px;height:12px;background:%s;display:inline-block;border:1px solid #999"></i>'
                   '<span style="font-size:10px">%s (%s)</span></div>' % (col, nom, _rango))
_leyenda_ipm = (
    '<div id="ipm-leyenda" style="position:fixed;bottom:24px;right:24px;z-index:9999;'
    'background:' + BG + ';padding:8px 10px;border-radius:8px;box-shadow:0 2px 8px rgba(0,0,0,.25);'
    'font-family:system-ui,Segoe UI,sans-serif;color:' + OKI_GRIS + ';max-width:215px">'
    '<div style="font-weight:700;font-size:11px;margin-bottom:4px">IPM - Pobreza (nivel)</div>'
    + _items_ley +
    '<div style="font-size:9px;color:#888;margin-top:4px">Mayor valor = mas pobreza. Umbral de pobreza: 33,3</div></div>')
m_ipm.get_root().html.add_child(folium.Element(_leyenda_ipm))
folium.LayerControl(collapsed=False, position="topright").add_to(m_ipm)
print("Mapa de IPM construido. Manzanas en zona:", len(ipm_z))

## 6. Panel colapsable: resumen + variables de privacion

In [ ]:
# 6a. Panel flotante colapsable: resumen IPM + grafica de barras de las 15 variables de
#      privacion (promedio en la zona). El IPM es un corte censal (sin serie de tiempo),
#      por eso se grafican las privaciones, no una evolucion anual.
_ipm_prom = float(ipm_z["ipm"].mean())
_ipm_max = float(ipm_z["ipm"].max())
_n_mz = int(len(ipm_z))
# Promedio por variable de privacion, ordenado de mayor a menor.
_priv = sorted([(VAR_LABEL[v], float(ipm_z[v].mean())) for v in VARS_PRIV],
               key=lambda x: x[1], reverse=True)
_labels = [n for n, _ in _priv]
_data = [round(val, 1) for _, val in _priv]
# Color unico Okabe-Ito morado (coherente con la coropleta).
_resumen_items = (
    '<div style="margin:2px 0 4px 0">Manzanas en zona: <b>%d</b> &nbsp;|&nbsp; '
    'IPM promedio: <b>%.1f</b> &nbsp;|&nbsp; IPM maximo: <b>%.1f</b></div>'
    % (_n_mz, _ipm_prom, _ipm_max)
)
# Distribucion de manzanas por nivel IPM (escala de 4 niveles, mapa tipo calor).
_cont_niv = {nom: 0 for (_lo, _hi, nom, _c) in IPM_NIVELES}
for _v in ipm_z['ipm']:
    _cont_niv[_cat_nombre(_v)] += 1
_resumen_items += '<div style="margin:4px 0 2px 0;font-weight:700">Manzanas por nivel:</div>'
for (_lo, _hi, nom, col) in IPM_NIVELES:
    _resumen_items += ('<div style="display:flex;align-items:center;gap:6px;margin:1px 0">'
        '<i style="width:12px;height:12px;background:%s;display:inline-block;border-radius:2px;border:1px solid #777"></i>'
        '<span>%s: <b>%d</b></span></div>' % (col, nom, _cont_niv[nom]))
# Tabla de respaldo.
_filas = "".join('<tr><td style="padding:1px 6px">%s</td>'
                 '<td style="padding:1px 6px;text-align:right"><b>%.1f%%</b></td></tr>'
                 % (n, val) for n, val in _priv)
_tabla_html = ('<table style="font-size:11px;border-collapse:collapse;width:100%">'
               '<tr style="border-bottom:1px solid #ccc">'
               '<th style="padding:1px 6px;text-align:left">Privacion</th>'
               '<th style="padding:1px 6px;text-align:right">Promedio</th></tr>'
               + _filas + '</table>')
_chart_data = json.dumps({"labels": _labels, "data": _data}, ensure_ascii=False)
_panel = """
<div id="ipm-panel" style="position:fixed;top:20px;left:20px;z-index:9999;
     background:__BG__;max-width:480px;width:min(480px,46vw);
     padding:10px 12px;border-radius:10px;box-shadow:0 2px 10px rgba(0,0,0,.3);
     font-family:system-ui,Segoe UI,sans-serif;font-size:12px;color:__GRIS__">
  <div style="font-weight:700;margin-bottom:4px">IPM - indice de pobreza multidimensional</div>
  __RESUMEN__
  <details style="margin-top:6px" open>
    <summary style="cursor:pointer;font-weight:600;color:__AZUL__;outline:none">
      Variables de privacion (promedio de la zona) / Ocultar</summary>
    <div style="margin-top:8px">
      <div style="font-size:9px;color:#666;margin-bottom:4px">% de hogares/personas con cada privacion (promedio de las manzanas de la zona).</div>
      <div style="position:relative;height:300px;width:100%"><canvas id="ipm-chart"></canvas></div>
      <div id="ipm-fallback" style="display:none;max-height:220px;overflow:auto">__TABLA__</div>
    </div>
  </details>
</div>
<script src="https://cdn.jsdelivr.net/npm/chart.js"></script>
<script>
(function(){
  var DATA = __DATA__;
  function dibujar(){
    var cv = document.getElementById('ipm-chart');
    var fb = document.getElementById('ipm-fallback');
    if (typeof Chart === 'undefined' || !cv){
      if (cv) cv.style.display='none';
      if (fb) fb.style.display='block';
      return;
    }
    try {
      new Chart(cv.getContext('2d'), {
        type: 'bar',
        data: {labels: DATA.labels,
          datasets: [{label: 'Privacion (%)', data: DATA.data,
                      backgroundColor: '#c51b8a', borderColor: '#7a0177', borderWidth: 1}]},
        options: {indexAxis: 'y', responsive:true, maintainAspectRatio:false,
          plugins:{legend:{display:false}},
          scales:{x:{beginAtZero:true, max:100, ticks:{font:{size:9}},
                     title:{display:true,text:'% de incidencia',font:{size:9}}},
                  y:{ticks:{font:{size:9}}}}}
      });
    } catch(e){
      if (cv) cv.style.display='none';
      if (fb) fb.style.display='block';
    }
  }
  if (document.readyState === 'complete'){ setTimeout(dibujar, 300); }
  else { window.addEventListener('load', function(){ setTimeout(dibujar, 300); }); }
})();
</script>
"""
_panel = (_panel.replace("__BG__", BG).replace("__GRIS__", OKI_GRIS)
                .replace("__AZUL__", OKI_AZUL).replace("__RESUMEN__", _resumen_items)
                .replace("__TABLA__", _tabla_html).replace("__DATA__", _chart_data))
m_ipm.get_root().html.add_child(folium.Element(_panel))
print("Panel colapsable inyectado (resumen IPM + grafica de privaciones).")

## 7. Exportacion del HTML

In [ ]:
agregar_export(m_ipm, "IPM - Pobreza Multidimensional", "ipm")
html_ipm=os.path.join(MAPAS_DIR,"mapa_ipm.html")
m_ipm.save(html_ipm)
print("Mapa de IPM exportado:",html_ipm)
m_ipm